#  Scalable Visualization and Explainability of Synthetic Datasets

In [ ]:
%load_ext jupyter_black

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
import seaborn as sns
import time
import warnings
import logging
import os

from matplotlib.ticker import ScalarFormatter
from scipy.stats import ks_2samp, mannwhitneyu, ttest_ind
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.manifold import (
    # TSNE,
    LocallyLinearEmbedding,
    Isomap,
    MDS,
    SpectralEmbedding,
)


from sklearn.linear_model import LinearRegression

from umap import UMAP

from openTSNE import TSNE

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# from tsnecuda import TSNE as TSNE_GPU

# from cuml.manifold.umap import UMAP
# from cuml.manifold import TSNE

from vizdataquality import (
    calculate as vdqc,
    datasets as vdqd,
    plot as vdqp,
    report as vdqr,
)

In [ ]:
pip install -q plotly

In [ ]:
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)

In [ ]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [ ]:
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
sns.set(
    context="notebook",
    rc={"figure.figsize": (12, 10)},
    palette=sns.color_palette("tab10", 10),
)

In [ ]:
!python --version

In [ ]:
start = time.time()

## Data Understanding

### Real Data - Insurance

In [ ]:
insurance_rdf = pd.read_csv("../data/raw/data/insurance/real/insurance.csv")

In [ ]:
insurance_rdf.head()

### Synthetic Data  - Insurance

In [ ]:
logging.info("Starting to read the insurance dataset.")

start_time = time.time()
try:
    insurance_sdf = pd.read_csv(
        "../data/raw/data/insurance/synth/insurance_1M.csv", index_col=0
    )
    elapsed = time.time() - start_time
    logging.info(
        f"Successfully read insurance dataset. Shape: {insurance_sdf.shape}. Time taken: {elapsed:.2f} seconds."
    )
except Exception as e:
    logging.error("Failed to read the insurance dataset.", exc_info=True)

In [ ]:
insurance_sdf = pd.read_csv("../data/raw/data/insurance/synth/insurance_1M.csv")

In [ ]:
insurance_sdf.head()

In [ ]:
insurance_sdf1 = pd.read_csv("../data/raw/data/insurance/synth/insurance_100K.csv")

In [ ]:
insurance_sdf1.head()

### Descriptive Statistics

#### Real Data

In [ ]:
insurance_rdf.describe()

#### Synthethic Data

In [ ]:
# 1M
insurance_sdf.describe()

In [ ]:
# 100K
insurance_sdf1.describe()

## Data Validation & Profiling

In [ ]:
# data validation real data
vdqc.calc(insurance_rdf)

In [ ]:
## Data validation synthetic data
vdqc.calc(insurance_sdf)

In [ ]:
# data validation synthetic data
vdqc.calc(insurance_sdf1)

## EDA

In [ ]:
insurance_rdf["AGE"] = insurance_rdf["AGE"].astype("float")

### Violin Plot Real  Data

In [ ]:
def violin_plot(data, save_path=None, **kwargs):

    numeric_data = data.select_dtypes("float")
    fig, ax = plt.subplots(1, len(numeric_data.columns), **kwargs)

    for i, col in enumerate(numeric_data.columns):
        ax[i].violinplot(numeric_data[col], showmedians=True, orientation="vertical")
        ax[i].set_title(f"Distribution of {col}")
        ax[i].set_ylabel("Value")
        ax[i].set_xticks([1])
        ax[i].set_xticklabels([col])

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f"{save_path}_violin_plot", dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
violin_plot(
    insurance_rdf[["AGE", "BMI", "CHARGES"]],
    save_path="../reports/figures/distribution of continuous variables_real",
    figsize=(6, 3),
)

### Violin Plot Synthetic  Data - 1M rows

In [ ]:
violin_plot(
    insurance_sdf[["AGE", "BMI", "CHARGES"]],
    save_path="../reports/figures/distribution of continuous variables_synth",
    figsize=(6, 3),
)

### Violin Plot 100k

In [ ]:
violin_plot(insurance_sdf1[["AGE", "BMI", "CHARGES"]], figsize=(6, 3))

### Histogram plot

In [ ]:
def histogram_plot(data, num_rows, num_columns, bins=30, save_path=None):

    numeric_data = data.select_dtypes("float")
    fig, ax = plt.subplots(
        num_rows, num_columns, figsize=(5 * num_columns, 3 * num_rows)
    )
    ax = ax.flatten()

    for i, col in enumerate(numeric_data.columns):
        sns.histplot(numeric_data[col], ax=ax[i], kde=True, bins=bins)
        ax[i].set_title(f"Distribution of {col}")
        ax[i].set_ylabel("Value")

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f"{save_path}_hist_plot", dpi=300, bbox_inches="tight")

    plt.show()

#### Real Data

In [ ]:
histogram_plot(
    data=insurance_rdf[["AGE", "BMI", "CHARGES"]],
    num_rows=1,
    num_columns=3,
    save_path="../reports/figures/distribution of continuous variables_real",
)

#### Synthethic Data 1M

In [ ]:
histogram_plot(
    data=insurance_sdf[["AGE", "BMI", "CHARGES"]],
    num_rows=1,
    num_columns=3,
    save_path="../reports/figures/distribution of continuous variables_synth",
)

#### Synthethic Data 100k

In [ ]:
histogram_plot(
    data=insurance_sdf1[["AGE", "BMI", "CHARGES"]], num_rows=1, num_columns=3
)

### Categorical Plots

In [ ]:
# def cat_plot(df, save_path=None, **kwargs):

#     cat = df.select_dtypes(include=["int", "object"])
#     fig, ax = plt.subplots(1, len(cat.columns), **kwargs)

#     for i, col in enumerate(cat):
#         count = cat[col].value_counts().reset_index()
#         count.columns = [col, "count"]
#         sns.barplot(x=col, y="count", data=count, ax=ax[i])
#         ax[i].set_title(f"Distribution of {col}")
#         ax[i].set_ylabel("Value")
#         ax[i].tick_params(axis="x", rotation=45)

#     if save_path:
#         os.makedirs(os.path.dirname(save_path), exist_ok=True)
#         plt.savefig(f"{save_path}_bar_plot", dpi=300, bbox_inches="tight")

#     plt.tight_layout()
#     plt.show()

In [ ]:
def cat_plot(df, save_path=None, **kwargs):
    # Select object columns
    obj_cols = df.select_dtypes(include="object").columns.tolist()

    # Select int columns with < 10 unique values
    int_cols = [
        col
        for col in df.select_dtypes(include="int").columns
        if df[col].nunique() <= 10
    ]

    # Combine into one list of categorical-like columns
    cat_cols = obj_cols + int_cols

    if not cat_cols:
        print("No categorical or low-cardinality integer columns to plot.")
        return

    fig, ax = plt.subplots(1, len(cat_cols), **kwargs)

    if len(cat_cols) == 1:
        ax = [ax]

    for i, col in enumerate(cat_cols):
        count = df[col].value_counts().reset_index()
        count.columns = [col, "count"]
        sns.barplot(x=col, y="count", data=count, ax=ax[i])
        ax[i].set_title(f"Distribution of {col}")
        ax[i].set_ylabel("Count")
        ax[i].tick_params(axis="x", rotation=45)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f"{save_path}_bar_plot.png", dpi=300, bbox_inches="tight")

    plt.tight_layout()
    plt.show()

####  Real Data

In [ ]:
cat_plot(
    insurance_rdf[["SEX", "CHILDREN", "SMOKER", "REGION"]],
    save_path="../reports/figures/distribution of cat variables_real",
    figsize=(8, 3),
)

#### Synthethic Data - 1M

In [ ]:
cat_plot(
    insurance_sdf,
    save_path="../reports/figures/distribution of cat variables_synth",
    figsize=(8, 3),
)

#### Synthethic Data - 100K

In [ ]:
cat_plot(insurance_sdf1, figsize=(8, 3))

### Comparison of Distributions between Real and Synthethic Data

In [ ]:
def plot_distribution_comparison(
    df1,
    df2,
    column="VALUE",
    label1="Dataset 1",
    label2="Dataset 2",
    save_path=None,
    **kwargs,
):
    figsize = kwargs.get("figsize", (8, 5))
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # Histogram comparison
    sns.histplot(
        df1[column],
        bins=30,
        color="blue",
        label=label1,
        stat="density",
        alpha=0.5,
        element="step",
        ax=ax,
    )
    sns.histplot(
        df2[column],
        bins=30,
        color="red",
        label=label2,
        stat="density",
        alpha=0.5,
        element="step",
        ax=ax,
    )
    ax.set_xlabel(column)
    ax.set_ylabel("Density")
    ax.set_title(f"Histogram Comparison of {column}")
    ax.legend()

    # --- Statistical Tests ---
    data1 = df1[column].dropna()
    data2 = df2[column].dropna()

    # Kolmogorov-Smirnov Test
    ks_stat, ks_p = ks_2samp(data1, data2)

    # Mann-Whitney U Test
    mw_stat, mw_p = mannwhitneyu(data1, data2, alternative="two-sided")

    # T-test
    t_stat, t_p = ttest_ind(data1, data2, equal_var=False)

    # Annotate plot with p-values
    textstr = "\n".join(
        (f"KS p = {ks_p:.3g}", f"MW U p = {mw_p:.3g}", f"T-test p = {t_p:.3g}")
    )

    ax.text(
        0.98,
        0.95,
        textstr,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.6),
    )

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(f"{save_path}_hist_diff", dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
for col in insurance_rdf.select_dtypes("float").columns:
    plot_distribution_comparison(
        insurance_rdf,
        insurance_sdf,
        column=col,
        label1="Real",
        label2="Synthetic",
        save_path=f"../reports/figures/{col}_hist_diff",
    )

## Data Preprocessing

In [ ]:
insurance_rdf.dtypes

In [ ]:
insurance_rdf = insurance_rdf.reset_index(drop=True)
insurance_sdf = insurance_sdf.reset_index(drop=True)

In [ ]:
# convert categorical data to categories
insurance_rdf[["SEX", "SMOKER", "REGION"]] = insurance_sdf[
    ["SEX", "SMOKER", "REGION"]
].astype("category")

In [ ]:
insurance_sdf[["SEX", "SMOKER", "REGION"]] = insurance_sdf[
    ["SEX", "SMOKER", "REGION"]
].astype("category")

In [ ]:
insurance_sdf1[["SEX", "SMOKER", "REGION"]] = insurance_sdf[
    ["SEX", "SMOKER", "REGION"]
].astype("category")

In [ ]:
insurance_rdf.dtypes

In [ ]:
insurance_sdf.dtypes

### One Hot Encoding

In [ ]:
def encode_categorical_features(df):
    """
    One-hot encodes all categorical (object-type) columns in the given DataFrame.

    Parameters:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A new DataFrame with categorical features one-hot encoded.
    """
    # Select categorical columns
    categorical_columns = df.select_dtypes(include="category").columns

    # Create column transformer for one-hot encoding
    categorical_transformer = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(drop="first"), categorical_columns)],
        remainder="passthrough",
    )

    # Create and apply pipeline
    pipeline = Pipeline(steps=[("preprocess", categorical_transformer)])
    processed_arr = pipeline.fit_transform(df)

    # Get feature names
    ohe = pipeline.named_steps["preprocess"].named_transformers_["cat"]
    encoded_feature_names = ohe.get_feature_names_out(categorical_columns)
    other_feature_names = [col for col in df.columns if col not in categorical_columns]
    all_feature_names = list(encoded_feature_names) + list(other_feature_names)

    return pd.DataFrame(processed_arr, columns=all_feature_names)

In [ ]:
processed_rdf = encode_categorical_features(insurance_rdf)

In [ ]:
output_path = "../data/processed/processed_real_df.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

processed_rdf.to_csv(output_path, index=False)

In [ ]:
processed_rdf.head()

In [ ]:
processed_sdf = encode_categorical_features(insurance_sdf)

In [ ]:
output_path = "../data/processed/processed_synth_df.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

processed_sdf.to_csv(output_path, index=False)

In [ ]:
processed_sdf.head()

## Dimensionality Reduction Algorithms

In [ ]:
def compare_embeddings(
    real_data,
    synthetic_data,
    algorithm,
    n_components=2,
    n_real_samples=None,
    n_synth_samples=None,
    random_state=42,
    save_path=None,
    **kwargs,
):
    """
    Compare real and synthetic data using t-SNE (openTSNE) or UMAP.
    Both algorithms are fit on real data only and transform both datasets separately.
    """
    algorithm = algorithm.lower()
    if algorithm not in {"tsne", "umap"}:
        raise ValueError(f"Unsupported algorithm: {algorithm!r}")

    real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
    synth = (
        synthetic_data.values
        if isinstance(synthetic_data, pd.DataFrame)
        else synthetic_data
    )

    if n_real_samples is not None and n_real_samples > len(real):
        raise ValueError(
            f"Requested {n_real_samples} real samples, but only {len(real)} available."
        )
    if n_synth_samples is not None and n_synth_samples > len(synth):
        raise ValueError(
            f"Requested {n_synth_samples} synthetic samples, but only {len(synth)} available."
        )

    rng = np.random.default_rng(random_state)
    real_n = n_real_samples or len(real)
    synth_n = n_synth_samples or len(synth)

    real_idx = rng.choice(len(real), real_n, replace=False)
    synth_idx = rng.choice(len(synth), synth_n, replace=False)
    real_sampled = real[real_idx]
    synth_sampled = synth[synth_idx]

    real_sampled = np.nan_to_num(real_sampled.astype(np.float32))
    synth_sampled = np.nan_to_num(synth_sampled.astype(np.float32))

    start = time.time()

    if algorithm == "tsne":
        tsne = TSNE(
            n_components=n_components,
            random_state=random_state,
            verbose=True,
            **kwargs,
        )
        tsne_embedding = tsne.fit(real_sampled)
        embedding_real = np.array(tsne_embedding.transform(real_sampled))
        embedding_synth = np.array(tsne_embedding.transform(synth_sampled))
    else:
        umap_model = UMAP(
            n_components=n_components, random_state=random_state, **kwargs
        )
        umap_model.fit(real_sampled)
        embedding_real = umap_model.transform(real_sampled)
        embedding_synth = umap_model.transform(synth_sampled)

    elapsed = time.time() - start

    df_real = pd.DataFrame(
        embedding_real, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )
    df_synth = pd.DataFrame(
        embedding_synth, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_real[f"{algorithm.upper()}_1"],
            y=df_real[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Real (n={real_n})",
            marker=dict(size=4, opacity=0.3),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df_synth[f"{algorithm.upper()}_1"],
            y=df_synth[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Synthetic (n={synth_n})",
            marker=dict(size=2, opacity=0.3),
        )
    )

    fig.update_layout(
        title=f"{algorithm.upper()} ({n_components}D) — {elapsed:.2f}s",
        xaxis_title=f"{algorithm.upper()}_1",
        yaxis_title=f"{algorithm.upper()}_2",
        width=800,
        height=600,
        legend_title="Data Type (sample size)",
    )

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.write_html(save_path)

    fig.show()
    return elapsed

###  Gaussian Noise

- Adding gaussian noise to the synthethic data

In [ ]:
def add_statistical_noise(df, noise_mean=0, shift_factor=0.5, random_state=None):
    """
    Adds Gaussian noise to numerical columns, shifting them using a fraction of their own std.
    Columns originally bounded at 0 are clipped to preserve non-negativity.

    Parameters:
    - df: Original DataFrame
    - shift_factor: Fraction of std used for both mean shift and noise std
    - random_state: Optional seed

    Returns:
    - df_noisy: Noisy, statistically shifted DataFrame
    """
    rng = np.random.default_rng(random_state)
    df_noisy = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        original_min = df[col].min()
        original_dtype = df[col].dtype

        std = df[col].std()
        noise_std = shift_factor * std

        noise = rng.normal(loc=noise_mean, scale=noise_std, size=df[col].shape)
        df_noisy[col] = df[col] + noise

        # Clip if original values were strictly non-negative
        if original_min >= 0:
            df_noisy[col] = df_noisy[col].clip(lower=0)

        # If column had only integer values, round it
        if pd.api.types.is_integer_dtype(original_dtype) or (df[col] % 1 == 0).all():
            df_noisy[col] = df_noisy[col].round()

    return df_noisy

In [ ]:
noisy_sdf = add_statistical_noise(
    processed_sdf[["AGE", "BMI", "CHILDREN", "CHARGES"]],
    shift_factor=50,
)

In [ ]:
noisy_sdf.describe()

In [ ]:
noisy_synth_df = pd.concat([processed_sdf.iloc[:, :5], noisy_sdf], axis=1)

In [ ]:
output_path = "../data/processed/noisy_synth.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

noisy_synth_df.to_csv(output_path, index=False)

In [ ]:
noisy_synth_df.head(1)

### Interpretabiltiy

In [ ]:
real_sample_sizes = [1000, 1100, 1200, 1300]
synth_sample_sizes = [1000, 2000, 5000, 10000]

In [ ]:
runs_per_size = 3
times = {}
algorithm = "tsne"

In [ ]:
for real_size, synth_size in zip(real_sample_sizes, synth_sample_sizes):
    print(
        f"\nRunning {runs_per_size} times for sample sizes — Real: {real_size}, Synthetic: {synth_size}"
    )
    key = real_size + synth_size
    times[key] = []

    for run in range(runs_per_size):
        print(f"  Run {run + 1}...")
        try:
            elapsed_time = compare_embeddings(
                processed_rdf,
                processed_sdf,
                algorithm=algorithm,
                n_real_samples=real_size,
                n_synth_samples=synth_size,
                # n_neighbors=15,
                random_state=run,
                save_path=f"../reports/figures/{key}_run_{run}_embedding_plot.html",
            )
            times[key].append(elapsed_time)
        except ValueError as e:
            print(f"  Skipping due to error: {e}")

In [ ]:
# dataset_sizes = [1000, 2000, 5000, 10000, 15000, 20000, 25000, 30000]

In [ ]:
# runs_per_size = 5
# times = {}

In [ ]:
# for size in dataset_sizes:
#     print(f"\nRunning {runs_per_size} times for sample size {size}")
#     times[size] = []

#     for run in range(runs_per_size):
#         print(f"  Run {run + 1}...")
#         elapsed_time = compare_embeddings(
#             processed_rdf,
#             noisy_synth_df,
#             algorithm="umap",
#             n_real=size,
#             n_neighbors=15,
#             random_state=run,
#             save_path=f"../reports/figures/size_{size}_run_{run}_embedding_plot.html",
#         )
#         times[size].append(elapsed_time)

### Performance

In [ ]:
# all_algorithms = [
#     UMAP(),
#     LocallyLinearEmbedding(),
#     SpectralEmbedding(),
#     Isomap(n_neighbors=10),
#     TSNE(),
#     # MDS(),
# ]

In [ ]:
# def benchmark_algorithms(algorithms, data, n_runs=5, verbose=False):
#     """
#     Runs each algorithm multiple times on the data and returns a DataFrame
#     with runtime and output data per run.

#     Parameters:
#         algorithms: list of algorithms
#         data: input data (DataFrame)
#         n_runs: number of times to run each algorithm

#     Returns:
#         performance_df: DataFrame with columns: algorithm, run, time, data
#     """
#     records = []

#     for algorithm in algorithms:
#         alg_name = str(algorithm).split("()")[0]
#         if verbose:
#             print(f"{alg_name} running now")
#         for run in range(n_runs):
#             if verbose:
#                 print(f"current run {run}")
#             start_time = time.time()
#             fit = algorithm.fit_transform(data)
#             elapsed_time = time.time() - start_time
#             records.append(
#                 {"algorithm": alg_name, "run": run + 1, "time": elapsed_time, "data": fit}
#             )

#     performance_df = pd.DataFrame(records)
#     return performance_df

In [ ]:
# start = time.time()
# performance_df = benchmark_algorithms(
#     all_algorithms, processed_insurance_rdf, n_runs=5, verbose=True
# )
# print(f"Time to run benchmark algorithm", time.time() - start)

In [ ]:
# sns.lineplot(data=performance_df, x="run", y="time", hue="algorithm", marker="o")
# plt.title("Runtime per Run for Each Algorithm")
# plt.ylabel("Time (s)")
# plt.xlabel("Run")
# plt.legend(title="Algorithm")
# plt.grid(True)
# plt.show()

In [ ]:
dataset_sizes = np.array(list(times.keys()))
mean_times = np.array([np.mean(times[size]) for size in dataset_sizes])

In [ ]:
log_mean_times = np.log(mean_times)

# Fit linear regression on mean times
X = dataset_sizes.reshape(-1, 1)
y = log_mean_times

model = LinearRegression()
model.fit(X, y)

# Prediction line
x_line = np.linspace(min(dataset_sizes), max(dataset_sizes), 500).reshape(-1, 1)
y_line = model.predict(x_line)

In [ ]:
# Scatter points
scatter = go.Scatter(
    x=dataset_sizes,
    y=log_mean_times,
    mode="markers",
    marker=dict(size=6, opacity=0.6, color="blue"),
    name="Log(Mean Run Time)",
)

# Regression line
line = go.Scatter(
    x=x_line.flatten(),  # x_line was a 2D array
    y=y_line,
    mode="lines",
    line=dict(color="red", width=2),
    name="Linear Fit",
)

# Layout
layout = go.Layout(
    title="Linear Regression on Log-Transformed Mean Run Times",
    xaxis=dict(title="Dataset Size"),
    yaxis=dict(title="Log(Mean Run Time)"),
    width=800,
    height=500,
)

# Combine and plot
fig = go.Figure(data=[scatter, line], layout=layout)

fig.write_html("../reports/figures/log_runtime_regression_.html")

fig.show()

#### Predicted Time to process 100k - 1m rows

In [ ]:
x = np.linspace(100000, 1000000, 10)
y = model.predict(x.reshape(-1, 1))

In [ ]:
# Plot
plt.figure(figsize=(8, 5))
plt.scatter(x, y, marker="o", linestyle="-", color="blue", label="Run Time")
plt.xlabel("Dataset Size")
plt.ylabel("Run Time")
plt.title("Run Time vs Dataset Size")
plt.grid(True)
plt.legend()

plt.gca().xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
plt.ticklabel_format(style="plain", axis="x")

plt.tight_layout()

plt.savefig("../reports/figures/other_datasets_regression_.png")
plt.show()

In [ ]:
# plt.plot(dataset_sizes, times, marker="o")
# plt.title("t-SNE Runtime vs. Dataset Size")
# plt.xlabel("Number of Samples per Dataset (Real + Synthetic)")
# plt.ylabel("Elapsed Time (seconds)")
# plt.tight_layout()
# plt.show()

In [ ]:
# x_vals = []
# y_vals = []

# for size, run_times in times.items():
#     x_vals.extend([size] * len(run_times))
#     y_vals.extend(run_times)

In [ ]:
# x_vals = np.array(x_vals)
# y_vals = np.array(y_vals)

In [ ]:
# # figure
# plt.figure(figsize=(10, 6))

# # scatter of all individual runs
# sns.scatterplot(x=x_vals, y=y_vals, hue=x_vals, palette="viridis", alpha=0.2, s=1, legend=False)

# # line for each of the runs
# unique_sizes = np.unique(x_vals)
# for run_idx in range(runs_per_size):
#     run_times = y_vals[run_idx::runs_per_size]
#     plt.plot(
#         unique_sizes, run_times, marker="o", linewidth=1.5, alpha=0.6, label=f"Run {run_idx+1}"
#     )

# plt.title("U-Map Execution Time vs Dataset Size (Individual Runs Connected)")

# plt.xlabel("Dataset Size")
# plt.ylabel("Execution Time (seconds)")
# plt.grid(True)
# plt.legend(title="Run Number", loc="upper left", bbox_to_anchor=(1.02, 1))
# plt.tight_layout()
# plt.show()

### Adding Noise

In [ ]:
def generate_embedding_plot(
    real_data,
    synthetic_data,
    algorithm,
    n_components=2,
    n_real_samples=None,
    n_synth_samples=None,
    random_state=42,
    save_path=None,
    **kwargs,
):
    """
    Generate embedding plot comparing real and synthetic data using t-SNE or UMAP.
    Trains on real data only and transforms both datasets separately.

    Returns:
        elapsed_time: float
        fig: plotly.graph_objects.Figure
    """

    algorithm = algorithm.lower()
    if algorithm not in {"tsne", "umap"}:
        raise ValueError(f"Unsupported algorithm: {algorithm!r}")

    real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
    synth = (
        synthetic_data.values
        if isinstance(synthetic_data, pd.DataFrame)
        else synthetic_data
    )

    if n_real_samples is not None and n_real_samples > len(real):
        raise ValueError(
            f"Requested {n_real_samples} real samples, but only {len(real)} available."
        )
    if n_synth_samples is not None and n_synth_samples > len(synth):
        raise ValueError(
            f"Requested {n_synth_samples} synthetic samples, but only {len(synth)} available."
        )

    rng = np.random.default_rng(random_state)
    real_n = n_real_samples or len(real)
    synth_n = n_synth_samples or len(synth)

    real_idx = rng.choice(len(real), real_n, replace=False)
    synth_idx = rng.choice(len(synth), synth_n, replace=False)
    real_sampled = real[real_idx]
    synth_sampled = synth[synth_idx]

    real_sampled = np.nan_to_num(real_sampled.astype(np.float32))
    synth_sampled = np.nan_to_num(synth_sampled.astype(np.float32))

    start = time.time()

    if algorithm == "tsne":
        tsne = TSNE(
            n_components=n_components,
            random_state=random_state,
            verbose=False,
            **kwargs,
        )
        tsne_embedding = tsne.fit(real_sampled)
        embedding_real = tsne_embedding.transform(real_sampled)
        embedding_synth = tsne_embedding.transform(synth_sampled)
    else:
        umap_model = UMAP(
            n_components=n_components, random_state=random_state, **kwargs
        )
        umap_model.fit(real_sampled)
        embedding_real = umap_model.transform(real_sampled)
        embedding_synth = umap_model.transform(synth_sampled)

    elapsed = time.time() - start

    df_real = pd.DataFrame(
        embedding_real, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )
    df_synth = pd.DataFrame(
        embedding_synth, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_real[f"{algorithm.upper()}_1"],
            y=df_real[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Real (n={real_n})",
            marker=dict(size=5, opacity=0.5),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df_synth[f"{algorithm.upper()}_1"],
            y=df_synth[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Synthetic (n={synth_n})",
            marker=dict(size=3, opacity=0.5),
        )
    )

    fig.update_layout(
        title=f"{algorithm.upper()} ({n_components}D) — Shift={kwargs.get('shift_factor', 'N/A')} — {elapsed:.2f}s",
        xaxis_title=f"{algorithm.upper()}_1",
        yaxis_title=f"{algorithm.upper()}_2",
        width=800,
        height=600,
        legend_title="Data Type",
    )

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.write_html(save_path)

    return elapsed, fig

In [ ]:
# Config
shift_factors = np.round(np.linspace(5, 100, num=10))
real_sample_sizes = [1000]
synth_sample_sizes = [1000]
algorithm = "tsne"

frames = []
times = {}

for real_size, synth_size in zip(real_sample_sizes, synth_sample_sizes):
    print(f"\n--- Sample sizes: Real={real_size}, Synth={synth_size} ---")

    for i, shift in enumerate(shift_factors):
        print(f"\n=== Generating frame for shift_factor={shift} ===")

        # Add noise to synthetic data
        noisy_synth = add_statistical_noise(
            processed_sdf[["AGE", "BMI", "CHILDREN", "CHARGES"]],
            shift_factor=shift,
        )

        noisy_synth_df = pd.concat([processed_sdf.iloc[:, :5], noisy_synth], axis=1)

        key = f"real{real_size}_synth{synth_size}_shift{shift}"
        try:
            elapsed, fig = generate_embedding_plot(
                real_data=processed_rdf,
                synthetic_data=noisy_synth_df,
                algorithm=algorithm,
                n_real_samples=real_size,
                n_synth_samples=synth_size,
                random_state=42,
            )

            # Frame label includes both size and shift
            frame_name = f"R{real_size}_S{synth_size}_Shift{shift}"
            frame = go.Frame(data=fig.data, name=frame_name)
            frames.append(frame)
            times[key] = elapsed

        except ValueError as e:
            print(f"  Skipped due to error: {e}")
            times[key] = "Error"

In [ ]:
if frames:
    for frame in frames:
        shift_label = frame.name
        # Add a frame-specific title annotation
        frame.layout = go.Layout(
            annotations=[
                go.layout.Annotation(
                    text=shift_label,
                    showarrow=False,
                    xref="paper",
                    yref="paper",
                    x=0.5,
                    y=1.08,
                    font=dict(size=16),
                    align="center",
                )
            ]
        )

    anim_fig = go.Figure(
        data=frames[0].data,
        layout=go.Layout(
            title="Embedding Animation Across Shift Factors and Sample Sizes",
            updatemenus=[
                {
                    "type": "buttons",
                    "buttons": [
                        {
                            "label": "Play",
                            "method": "animate",
                            "args": [
                                None,
                                {
                                    "frame": {"duration": 4000, "redraw": True},
                                    "fromcurrent": True,
                                },
                            ],
                        },
                        {
                            "label": "Pause",
                            "method": "animate",
                            "args": [
                                [None],
                                {
                                    "frame": {"duration": 0, "redraw": False},
                                    "mode": "immediate",
                                },
                            ],
                        },
                    ],
                }
            ],
        ),
        frames=frames,
    )

    anim_fig.update_layout(
        width=800, height=600, xaxis_title="Component 1", yaxis_title="Component 2"
    )

    anim_fig.write_html("../reports/figures/embedding_animation_.html")
    anim_fig.show()
else:
    print("No animation frames generated.")

### Anomaly Detection

#### Local Outlier Factor

In [ ]:
def detect_and_highlight_problematic_clusters(
    real_data,
    synthetic_data,
    algorithm,
    n_components=2,
    n_real_samples=None,
    n_synth_samples=None,
    random_state=42,
    contamination="auto",
    n_neighbors_umap=15,
    n_neighbors_lof=20,
    save_path=None,
    **kwargs,
):
    """
    Detect and highlight problematic synthetic clusters (outliers) in synthetic data
    that are not consistent with the real data using t-SNE or UMAP with LOF (k-NN based).

    Args:
        real_data: Real data (e.g., pd.DataFrame or np.ndarray)
        synthetic_data: Synthetic data (e.g., pd.DataFrame or np.ndarray)
        algorithm: The dimensionality reduction algorithm to use ('tsne' or 'umap')
        n_components: Number of dimensions for embedding (usually 2 or 3)
        n_real_samples: Number of real samples to use for embedding
        n_synth_samples: Number of synthetic samples to use for embedding
        random_state: Random seed for reproducibility
        contamination: Proportion of outliers in the synthetic data
        save_path: Path to save the plot (if provided)
        **kwargs: Additional parameters for the embedding and anomaly algorithms

    Returns:
        problematic_synthetic_data: The problematic synthetic data points (as a DataFrame)
    """
    # Convert data to numpy if they are pandas DataFrames
    real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
    synth = (
        synthetic_data.values
        if isinstance(synthetic_data, pd.DataFrame)
        else synthetic_data
    )

    # Sample data
    rng = np.random.default_rng(random_state)
    real_n = n_real_samples or len(real)
    synth_n = n_synth_samples or len(synth)

    real_idx = rng.choice(len(real), real_n, replace=False)
    synth_idx = rng.choice(len(synth), synth_n, replace=False)
    real_sampled = real[real_idx]
    synth_sampled = synth[synth_idx]

    real_sampled = np.nan_to_num(real_sampled.astype(np.float32))
    synth_sampled = np.nan_to_num(synth_sampled.astype(np.float32))

    # Compute embeddings
    if algorithm == "tsne":
        tsne = TSNE(n_components=n_components, random_state=random_state, **kwargs)
        tsne_embedding = tsne.fit(real_sampled)
        embedding_real = np.array(tsne_embedding.transform(real_sampled))
        embedding_synth = np.array(tsne_embedding.transform(synth_sampled))
    else:
        umap_model = UMAP(
            n_components=n_components,
            random_state=random_state,
            n_neighbors=n_neighbors_umap,
            **kwargs,
        )
        umap_model.fit(real_sampled)
        embedding_real = umap_model.transform(real_sampled)
        embedding_synth = umap_model.transform(synth_sampled)

    # Apply LOF (k-NN based) for anomaly detection with n_neighbors_lof
    lof_detector = LocalOutlierFactor(
        n_neighbors=n_neighbors_lof, contamination=contamination
    )

    # Fit LOF to the real data and predict anomalies in the synthetic data
    anomaly_scores_synth = lof_detector.fit_predict(embedding_synth)
    problematic_synth = np.where(anomaly_scores_synth == -1)[0]  # Identifying outliers

    # Create DataFrames for visualization
    df_real = pd.DataFrame(
        embedding_real, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )
    df_synth = pd.DataFrame(
        embedding_synth, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )

    # Extract the problematic synthetic data points from the original synthetic dataset
    problematic_synthetic_data = synthetic_data.iloc[
        problematic_synth
    ]  # Use the original DataFrame

    # Plot the embeddings
    fig = go.Figure()

    # Real data plot (blue)
    fig.add_trace(
        go.Scatter(
            x=df_real[f"{algorithm.upper()}_1"],
            y=df_real[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Real (n={real_n})",
            marker=dict(size=5, opacity=0.5, symbol="circle", color="blue"),
        )
    )

    # Synthetic data plot (green)
    fig.add_trace(
        go.Scatter(
            x=df_synth[f"{algorithm.upper()}_1"],
            y=df_synth[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Synthetic (n={synth_n})",
            marker=dict(size=5, opacity=0.5, symbol="circle", color="#00bcd4"),
        )
    )

    # Highlight problematic synthetic points (outliers) with transparent fill and red border
    problematic_synth_x = df_synth.iloc[
        problematic_synth, 0
    ]  # x values for problematic points
    problematic_synth_y = df_synth.iloc[
        problematic_synth, 1
    ]  # y values for problematic points

    fig.add_trace(
        go.Scatter(
            x=problematic_synth_x,
            y=problematic_synth_y,
            mode="markers",
            name="Problematic Synthetic",
            marker=dict(
                size=10,
                color="rgba(0,0,0,0)",
                symbol="circle",
                line=dict(color="red", width=1),
                opacity=0.3,
            ),
        )
    )

    # Update layout and display
    fig.update_layout(
        title=f"{algorithm.upper()} ({n_components}D) — Highlighting Problematic Synthetic Clusters",
        xaxis_title=f"{algorithm.upper()}_1",
        yaxis_title=f"{algorithm.upper()}_2",
        width=800,
        height=600,
        legend_title="Data Type (sample size)",
    )

    # Save plot if path is provided
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.write_html(save_path)

    fig.show()

    # Return the problematic synthetic data points with the correct columns
    return problematic_synthetic_data

In [ ]:
problematic_data_points = detect_and_highlight_problematic_clusters(
    processed_rdf,
    noisy_synth_df,
    n_real_samples=1000,
    n_synth_samples=1000,
    algorithm="tsne",
)

In [ ]:
# Inspect the problematic synthetic data
print(problematic_data_points.shape)
problematic_data_points.head()

#### IsolationForest

In [ ]:
def detect_and_highlight_problematic_clusters(
    real_data,
    synthetic_data,
    algorithm,
    n_components=2,
    n_real_samples=None,
    n_synth_samples=None,
    random_state=None,
    contamination="auto",
    save_path=None,
    **kwargs,
):
    """
    Detect and highlight problematic synthetic clusters (outliers) in synthetic data
    that are not consistent with the real data using t-SNE or UMAP with Isolation Forest.

    Args:
        real_data: Real data (e.g., pd.DataFrame or np.ndarray)
        synthetic_data: Synthetic data (e.g., pd.DataFrame or np.ndarray)
        algorithm: The dimensionality reduction algorithm to use ('tsne' or 'umap')
        n_components: Number of dimensions for embedding (usually 2 or 3)
        n_real_samples: Number of real samples to use for embedding
        n_synth_samples: Number of synthetic samples to use for embedding
        random_state: Random seed for reproducibility
        contamination: Proportion of outliers in the synthetic data
        save_path: Path to save the plot (if provided)
        **kwargs: Additional parameters for the embedding and anomaly algorithms

    Returns:
        problematic_synthetic_data: The problematic synthetic data points (as a DataFrame)
    """
    # Convert data to numpy if they are pandas DataFrames
    real = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
    synth = (
        synthetic_data.values
        if isinstance(synthetic_data, pd.DataFrame)
        else synthetic_data
    )

    # Sample data
    rng = np.random.default_rng(random_state)
    real_n = n_real_samples or len(real)
    synth_n = n_synth_samples or len(synth)

    real_idx = rng.choice(len(real), real_n, replace=False)
    synth_idx = rng.choice(len(synth), synth_n, replace=False)
    real_sampled = real[real_idx]
    synth_sampled = synth[synth_idx]

    real_sampled = np.nan_to_num(real_sampled.astype(np.float32))
    synth_sampled = np.nan_to_num(synth_sampled.astype(np.float32))

    # Compute embeddings
    if algorithm == "tsne":
        tsne = TSNE(n_components=n_components, random_state=random_state, **kwargs)
        tsne_embedding = tsne.fit(real_sampled)
        embedding_real = np.array(tsne_embedding.transform(real_sampled))
        embedding_synth = np.array(tsne_embedding.transform(synth_sampled))
    else:
        umap_model = UMAP(
            n_components=n_components, random_state=random_state, **kwargs
        )
        umap_model.fit(real_sampled)
        embedding_real = umap_model.transform(real_sampled)
        embedding_synth = umap_model.transform(synth_sampled)

    # Apply Isolation Forest for anomaly detection
    anomaly_detector = IsolationForest(
        contamination=contamination, random_state=random_state
    )

    anomaly_detector.fit(embedding_real)
    anomaly_scores_synth = anomaly_detector.predict(embedding_synth)
    problematic_synth = np.where(anomaly_scores_synth == -1)[0]  # Identifying outliers

    # Create DataFrames for visualization
    df_real = pd.DataFrame(
        embedding_real, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )
    df_synth = pd.DataFrame(
        embedding_synth, columns=[f"{algorithm.upper()}_1", f"{algorithm.upper()}_2"]
    )

    # Extract the problematic synthetic data points from the original synthetic dataset
    problematic_synthetic_data = synthetic_data.iloc[
        problematic_synth
    ]  # Use the original DataFrame

    # Plot the embeddings
    fig = go.Figure()

    # Real data plot (blue)
    fig.add_trace(
        go.Scatter(
            x=df_real[f"{algorithm.upper()}_1"],
            y=df_real[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Real (n={real_n})",
            marker=dict(size=5, opacity=0.5, symbol="circle", color="blue"),
        )
    )

    # Synthetic data plot (green)
    fig.add_trace(
        go.Scatter(
            x=df_synth[f"{algorithm.upper()}_1"],
            y=df_synth[f"{algorithm.upper()}_2"],
            mode="markers",
            name=f"Synthetic (n={synth_n})",
            marker=dict(size=5, opacity=0.5, symbol="circle", color="#00bcd4"),
        )
    )

    # Highlight problematic synthetic points (outliers) with transparent fill and red border
    problematic_synth_x = df_synth.iloc[
        problematic_synth, 0
    ]  # x values for problematic points
    problematic_synth_y = df_synth.iloc[
        problematic_synth, 1
    ]  # y values for problematic points

    fig.add_trace(
        go.Scatter(
            x=problematic_synth_x,
            y=problematic_synth_y,
            mode="markers",
            name="Problematic Synthetic",
            marker=dict(
                size=10,
                color="rgba(0,0,0,0)",
                symbol="circle",
                line=dict(color="red", width=1),
                opacity=0.3,
            ),
        )
    )

    # Update layout and display
    fig.update_layout(
        title=f"{algorithm.upper()} ({n_components}D) — Highlighting Problematic Synthetic Clusters",
        xaxis_title=f"{algorithm.upper()}_1",
        yaxis_title=f"{algorithm.upper()}_2",
        width=800,
        height=600,
        legend_title="Data Type (sample size)",
    )

    # Save plot if path is provided
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.write_html(save_path)

    fig.show()

    # Return the problematic synthetic data points with the correct columns
    return problematic_synthetic_data

In [ ]:
problematic_synth_points = detect_and_highlight_problematic_clusters(
    processed_rdf,
    noisy_synth_df,
    n_real_samples=1000,
    n_synth_samples=1000,
    algorithm="tsne",
)

In [ ]:
print(problematic_synth_points.shape)
problematic_synth_points.head()

In [ ]:
# len(set(problematic_synth_points.index).intersection(problematic_data_points.index))

### 3D Plots

In [ ]:
# for size in dataset_sizes:
#     print(f"\nRunning {runs_per_size} times for sample size {size}")
#     times[size] = []

#     for run in range(runs_per_size):
#         print(f"  Run {run + 1}...")
#         elapsed_time = compare_embeddings(
#             processed_rdf,
#             processed_sdf,
#             algorithm="umap",
#             n_real=size,
#             n_neighbors=15,
#             n_components=3,
#             # random_state=run,
#         )
#         times[size].append(elapsed_time)

In [ ]:
# x_vals = []
# y_vals = []

# for size, run_times in times.items():
#     x_vals.extend([size] * len(run_times))
#     y_vals.extend(run_times)

In [ ]:
# x_vals = np.array(x_vals)
# y_vals = np.array(y_vals)

In [ ]:
# # figure
# plt.figure(figsize=(10, 6))

# # scatter of all individual runs
# sns.scatterplot(x=x_vals, y=y_vals, hue=x_vals, palette="viridis", alpha=0.2, s=1, legend=False)

# # line for each of the runs
# unique_sizes = np.unique(x_vals)
# for run_idx in range(runs_per_size):
#     run_times = y_vals[run_idx::runs_per_size]
#     plt.plot(
#         unique_sizes, run_times, marker="o", linewidth=1.5, alpha=0.6, label=f"Run {run_idx+1}"
#     )

# plt.title("U-Map Execution Time vs Dataset Size (Individual Runs Connected)")

# plt.xlabel("Dataset Size")
# plt.ylabel("Execution Time (seconds)")
# plt.grid(True)
# plt.legend(title="Run Number", loc="upper left", bbox_to_anchor=(1.02, 1))
# plt.tight_layout()
# plt.show()

In [ ]:
# dataset_sizes = np.array(list(times.keys()))
# mean_times = np.array([np.mean(times[size]) for size in dataset_sizes])

In [ ]:
# plt.figure(figsize=(10, 6))
# sns.lineplot(
#     x=dataset_sizes,
#     y=mean_times,
#     color="blue",
#     alpha=0.6,
# )

# plt.title("U-Map Mean Execution Time vs Dataset Size")
# plt.xlabel("Dataset Size")
# plt.ylabel("Execution Time (seconds)")
# plt.grid(True)


# plt.tight_layout()
# plt.show()

### GPU



#### Interpretability


In [ ]:
# def tsne_compare_gpu(
#     real_data,
#     synthetic_data,
#     perplexity=15,
#     learning_rate=10,
#     n_real=None,
#     n_synth=None,
#     random_state=42,
# ):
#     """
#     Compare real and synthetic data using t-SNE embedding and plot the result.
#     Automatically balances 50% real and 50% synthetic. If not enough real data,
#     fills the gap with synthetic data.
#     Returns: elapsed time in seconds.
#     """
#     rng = np.random.default_rng(random_state)

#     # Convert to NumPy arrays if needed
#     real_data = real_data.values if isinstance(real_data, pd.DataFrame) else real_data
#     synthetic_data = (
#         synthetic_data.values
#         if isinstance(synthetic_data, pd.DataFrame)
#         else synthetic_data
#     )

#     # Determine target sample size
#     target_size = (
#         n_real if n_real is not None else min(len(real_data), len(synthetic_data))
#     )
#     n_half = target_size // 2

#     # Select real samples (as many as possible up to n_half)
#     real_sample_size = min(len(real_data), n_half)
#     real_indices = rng.choice(len(real_data), real_sample_size, replace=False)
#     selected_real = real_data[real_indices]

#     # Select synthetic samples: at least n_half + any shortfall from real
#     synth_needed = target_size - real_sample_size
#     synth_sample_size = synth_needed
#     synth_indices = rng.choice(len(synthetic_data), synth_sample_size, replace=False)
#     selected_synth = synthetic_data[synth_indices]

#     # Combine data and labels
#     print(
#         f"Size of data - real: {len(selected_real)}, synthethic:{len(selected_synth)}"
#     )
#     combined_data = np.vstack([selected_real, selected_synth])
#     labels = np.array(
#         ["Real"] * len(selected_real) + ["Synthetic"] * len(selected_synth)
#     )

#     # Start timing
#     start = time.time()

#     tsne = TSNE_GPU(n_components=2, perplexity=perplexity, learning_rate=learning_rate)
#     embedding = tsne.fit_transform(combined_data)

#     elapsed_time = time.time() - start

#     # Plot
#     plt.figure(figsize=(8, 6))
#     for label in ["Real", "Synthetic"]:
#         idx = labels == label
#         plt.scatter(embedding[idx, 0], embedding[idx, 1], label=label, alpha=0.5, s=10)

#     plt.title(
#         f"t-SNE Comparison (Perplexity={perplexity}, LR={learning_rate}, Time={elapsed_time:.2f}s)",
#         weight="bold",
#     )
#     plt.xlabel("t-SNE 1")
#     plt.ylabel("t-SNE 2")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

#     return elapsed_time

In [ ]:
# # %%timeit -n 1 -r 1

# dataset_sizes = [100, 500, 1000, 2000, 3000, 5000, 10000, 15000, 20000]
# times = []

# for size in dataset_sizes:
#     print(f"Running t-SNE for total sample size {size}")
#     elapsed_time = tsne_compare_gpu(
#         processed_rdf,
#         processed_sdf,
#         n_real=size,
#         perplexity=15,
#         learning_rate=15,
#     )
#     times.append(elapsed_time)

#### Performance

In [ ]:
# plt.plot(dataset_sizes, times, marker="o")
# plt.title("t-SNE GPU Enabled Runtime vs. Dataset Size")
# plt.xlabel("Number of Samples per Dataset (Real + Synthetic)")
# plt.ylabel("Elapsed Time (seconds)")
# plt.tight_layout()
# plt.show()